# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/home/mishrmuk/ai/projects/tinyml-arduino/bin/python


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [7]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [8]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

# <-- Enter your code here <--#
X = df.drop(columns=["Class"])
y = df["Class"]

# Optional sanity check
print("Feature matrix shape:", X.shape)
print("Label vector shape:", y.shape)
print("Unique labels:", y.unique())

Feature matrix shape: (178, 13)
Label vector shape: (178,)
Unique labels: [0 1 2]


In [9]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

# <-- Enter your code here <--#
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

# Optional sanity check
print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

Training set shape: (124, 13)
Test set shape: (54, 13)


In [10]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

# <-- Enter your code here <--#
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Optional sanity check
print("Scaled training data shape:", X_train_scaled.shape)
print("Scaled test data shape:", X_test_scaled.shape)

Scaled training data shape: (124, 13)
Scaled test data shape: (54, 13)


In [11]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

# <-- Enter your code here <--#
y_train_cat = tf.keras.utils.to_categorical(
    y_train,
    num_classes=num_classes
)

y_test_cat = tf.keras.utils.to_categorical(
    y_test,
    num_classes=num_classes
)

# Optional sanity check
print("One-hot encoded training labels shape:", y_train_cat.shape)
print("One-hot encoded test labels shape:", y_test_cat.shape)

One-hot encoded training labels shape: (124, 3)
One-hot encoded test labels shape: (54, 3)


In [12]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

# <-- Enter your code here <--#

model = Sequential([
    Dense(64, activation='relu', input_shape=(num_features,)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                896       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 3)                 99        
                                                                 
Total params: 3075 (12.01 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [13]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

# <-- Enter your code here <--#
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=20,
    batch_size=8,
    validation_split=0.2
)

Epoch 1/20
13/13 [==============================] - 1s 24ms/step - loss: 0.9126 - accuracy: 0.7475 - val_loss: 0.6297 - val_accuracy: 0.8800
Epoch 2/20
13/13 [==============================] - 0s 9ms/step - loss: 0.6430 - accuracy: 0.9192 - val_loss: 0.4348 - val_accuracy: 0.9600
Epoch 3/20
13/13 [==============================] - 0s 8ms/step - loss: 0.4478 - accuracy: 0.9495 - val_loss: 0.2899 - val_accuracy: 1.0000
Epoch 4/20
13/13 [==============================] - 0s 8ms/step - loss: 0.3003 - accuracy: 0.9596 - val_loss: 0.2030 - val_accuracy: 1.0000
Epoch 5/20
13/13 [==============================] - 0s 8ms/step - loss: 0.2059 - accuracy: 0.9596 - val_loss: 0.1490 - val_accuracy: 1.0000
Epoch 6/20
13/13 [==============================] - 0s 9ms/step - loss: 0.1438 - accuracy: 0.9798 - val_loss: 0.1192 - val_accuracy: 1.0000
Epoch 7/20
13/13 [==============================] - 0s 7ms/step - loss: 0.1099 - accuracy: 0.9899 - val_loss: 0.1044 - val_accuracy: 1.0000
Epoch 8/20
13/13 [=

In [14]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

# <-- Enter your code here <--#
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test_cat)

print("\nTest Accuracy:", test_accuracy)

# Generate predictions
y_pred_probs = model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_probs, axis=1)

# Classification report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# Confusion matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

2/2 [==============================] - 0s 8ms/step - loss: 0.0350 - accuracy: 0.9815

Test Accuracy: 0.9814814925193787
2/2 [==============================] - 0s 5ms/step

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      0.95      0.98        21
           2       0.93      1.00      0.97        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


Confusion Matrix:

[[19  0  0]
 [ 0 20  1]
 [ 0  0 14]]


In [17]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

# <-- Enter your code here <--#
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the model
with open("model_base.tflite", "wb") as f:
    f.write(tflite_model)

# Print file size in KB
model_size_kb = os.path.getsize("model_base.tflite") / 1024

print(f"\nTFLite model size: {model_size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmptlicboe8/assets


INFO:tensorflow:Assets written to: /tmp/tmptlicboe8/assets



TFLite model size: 14.07 KB


2026-05-12 22:39:53.293449: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-12 22:39:53.293536: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-12 22:39:53.293762: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmptlicboe8
2026-05-12 22:39:53.294826: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-12 22:39:53.294846: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmptlicboe8
2026-05-12 22:39:53.298739: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-12 22:39:53.351567: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmptlicboe8
2026-05-12 22:39:53.364751: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 70888 m

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [20]:
def file_size_kb(filename):
    return os.path.getsize(filename) / 1024

def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.

    # <-- Enter your code here <--#
    tflite_model = converter.convert()

    with open(filename, "wb") as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    # <-- Enter your code here for TFLite inference <--#
    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_index = input_details[0]["index"]
    output_index = output_details[0]["index"]

    input_dtype = input_details[0]["dtype"]
    output_dtype = output_details[0]["dtype"]

    input_scale, input_zero_point = input_details[0]["quantization"]
    output_scale, output_zero_point = output_details[0]["quantization"]

    y_pred = []

    for sample in X_test:
        sample = sample.reshape(1, -1).astype(np.float32)

        # Quantize input if needed
        if input_dtype == np.int8:
            sample = sample / input_scale + input_zero_point
            sample = np.round(sample).astype(np.int8)
        else:
            sample = sample.astype(input_dtype)

        interpreter.set_tensor(input_index, sample)
        interpreter.invoke()

        output = interpreter.get_tensor(output_index)

        # Dequantize output if needed
        if output_dtype == np.int8:
            output = output_scale * (output.astype(np.float32) - output_zero_point)

        pred_class = np.argmax(output, axis=1)[0]
        y_pred.append(pred_class)

    y_true = np.argmax(y_test_cat, axis=1)

    # Step 4: Report results.
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")

    # <-- Enter your code here: print classification_report and confusion_matrix <--#
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")

    accuracy = np.mean(np.array(y_pred) == y_true)
    print(f"{quant_type.upper()} Accuracy: {accuracy:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

In [21]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

# <-- Enter your code here <--#
quantize_and_evaluate(model, X_test_scaled, y_test_cat, "dynamic", "model_dynamic.tflite")
quantize_and_evaluate(model, X_test_scaled, y_test_cat, "int8", "model_int8.tflite")
quantize_and_evaluate(model, X_test_scaled, y_test_cat, "float16", "model_float16.tflite")

INFO:tensorflow:Assets written to: /tmp/tmpf0nkyygk/assets


INFO:tensorflow:Assets written to: /tmp/tmpf0nkyygk/assets
2026-05-12 22:47:39.354514: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-12 22:47:39.354603: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-12 22:47:39.354871: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpf0nkyygk
2026-05-12 22:47:39.356158: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-12 22:47:39.356178: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpf0nkyygk
2026-05-12 22:47:39.359899: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-12 22:47:39.409888: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpf0nkyygk
2026-05-12 22:47:39.429200: I tensorflow/cc/saved_model/loader.cc:316] SavedModel


DYNAMIC TFLite model size: 8.17 KB

DYNAMIC TFLite model size: 8.17 KB
DYNAMIC Accuracy: 0.9815

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      0.95      0.98        21
           2       0.93      1.00      0.97        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


Confusion Matrix:
[[19  0  0]
 [ 0 20  1]
 [ 0  0 14]]
INFO:tensorflow:Assets written to: /tmp/tmp48gkmkpx/assets


INFO:tensorflow:Assets written to: /tmp/tmp48gkmkpx/assets
/home/mishrmuk/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-12 22:47:40.252572: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-12 22:47:40.252658: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-12 22:47:40.252960: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp48gkmkpx
2026-05-12 22:47:40.254147: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-12 22:47:40.254171: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp48gkmkpx
2026-05-12 22:47:40.257790: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.



INT8 TFLite model size: 5.74 KB

INT8 TFLite model size: 5.74 KB
INT8 Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54


Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]
INFO:tensorflow:Assets written to: /tmp/tmpnsq2y6gt/assets


INFO:tensorflow:Assets written to: /tmp/tmpnsq2y6gt/assets
2026-05-12 22:47:41.147858: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-12 22:47:41.147944: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-12 22:47:41.148209: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpnsq2y6gt
2026-05-12 22:47:41.150505: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-12 22:47:41.150553: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpnsq2y6gt
2026-05-12 22:47:41.154350: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-12 22:47:41.200451: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpnsq2y6gt
2026-05-12 22:47:41.213080: I tensorflow/cc/saved_model/loader.cc:316] SavedModel


FLOAT16 TFLite model size: 8.95 KB

FLOAT16 TFLite model size: 8.95 KB
FLOAT16 Accuracy: 0.9815

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      0.95      0.98        21
           2       0.93      1.00      0.97        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


Confusion Matrix:
[[19  0  0]
 [ 0 20  1]
 [ 0  0 14]]


## Problem 1 - Part (c)

### Pruning

In [22]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

# <-- Enter your code here <--#
batch_size = 8
epochs = 20

# Approximate total training steps
end_step = np.ceil((len(X_train_scaled) * 0.8) / batch_size).astype(np.int32) * epochs

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step
)

print("End step:", end_step)

End step: 260


In [23]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

# <-- Enter your code here <--#

pruned_model = Sequential([
    tfmot.sparsity.keras.prune_low_magnitude(
        Dense(64, activation='relu', input_shape=(num_features,)),
        pruning_schedule=pruning_schedule
    ),
    tfmot.sparsity.keras.prune_low_magnitude(
        Dense(32, activation='relu'),
        pruning_schedule=pruning_schedule
    ),
    tfmot.sparsity.keras.prune_low_magnitude(
        Dense(num_classes, activation='softmax'),
        pruning_schedule=pruning_schedule
    )
])

pruned_model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_dense_  (None, 64)                1730      
 3 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 32)                4130      
 4 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 3)                 197       
 5 (PruneLowMagnitude)                                           
                                                                 
Total params: 6057 (23.67 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 2982 (11.66 KB)
_________________________________________________________________


In [24]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

# <-- Enter your code here <--#
pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep()
]

history_pruned = pruned_model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    callbacks=callbacks
)

Epoch 1/10
13/13 [==============================] - 3s 22ms/step - loss: 0.9374 - accuracy: 0.6263 - val_loss: 0.8532 - val_accuracy: 0.6800
Epoch 2/10
13/13 [==============================] - 0s 7ms/step - loss: 0.6792 - accuracy: 0.8283 - val_loss: 0.6524 - val_accuracy: 0.8000
Epoch 3/10
13/13 [==============================] - 0s 7ms/step - loss: 0.5038 - accuracy: 0.8990 - val_loss: 0.5005 - val_accuracy: 0.8800
Epoch 4/10
13/13 [==============================] - 0s 9ms/step - loss: 0.3744 - accuracy: 0.9394 - val_loss: 0.3866 - val_accuracy: 0.9200
Epoch 5/10
13/13 [==============================] - 0s 7ms/step - loss: 0.2781 - accuracy: 0.9596 - val_loss: 0.2822 - val_accuracy: 0.9600
Epoch 6/10
13/13 [==============================] - 0s 8ms/step - loss: 0.2084 - accuracy: 0.9596 - val_loss: 0.2177 - val_accuracy: 1.0000
Epoch 7/10
13/13 [==============================] - 0s 8ms/step - loss: 0.1602 - accuracy: 0.9697 - val_loss: 0.1716 - val_accuracy: 1.0000
Epoch 8/10
13/13 [=

In [25]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

# <-- Enter your code here <--#
stripped_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_pruned_model)
pruned_tflite_model = converter.convert()

with open("model_pruned.tflite", "wb") as f:
    f.write(pruned_tflite_model)

pruned_model_size_kb = os.path.getsize("model_pruned.tflite") / 1024

print(f"Pruned TFLite model size: {pruned_model_size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmpda1pd8ip/assets


INFO:tensorflow:Assets written to: /tmp/tmpda1pd8ip/assets
2026-05-12 22:56:20.433478: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-12 22:56:20.433575: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.


Pruned TFLite model size: 14.14 KB


2026-05-12 22:56:20.434103: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpda1pd8ip
2026-05-12 22:56:20.434919: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-12 22:56:20.434938: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpda1pd8ip
2026-05-12 22:56:20.437557: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-12 22:56:20.463619: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpda1pd8ip
2026-05-12 22:56:20.473704: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 39599 microseconds.


In [26]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
y_pred_probs = stripped_pruned_model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_probs, axis=1)

# True labels
y_true = np.argmax(y_test_cat, axis=1)

# Accuracy
accuracy = np.mean(y_pred == y_true)
print("Pruned Model Accuracy:", accuracy)

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

2/2 [==============================] - 0s 6ms/step
Pruned Model Accuracy: 0.9814814814814815

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        19
           1       1.00      1.00      1.00        21
           2       1.00      0.93      0.96        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 1  0 13]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [27]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

# <-- Enter your code here <--#
student_model = Sequential([
    Dense(32, activation='relu', input_shape=(num_features,)),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax')
])

student_model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_6 (Dense)             (None, 32)                448       
                                                                 
 dense_7 (Dense)             (None, 16)                528       
                                                                 
 dense_8 (Dense)             (None, 3)                 51        
                                                                 
Total params: 1027 (4.01 KB)
Trainable params: 1027 (4.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [28]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

# <-- Enter your code here <--#
teacher_soft_labels = model.predict(X_train_scaled)

print("Teacher soft labels shape:", teacher_soft_labels.shape)

4/4 [==============================] - 0s 3ms/step
Teacher soft labels shape: (124, 3)


In [30]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

# <-- Enter your code here <--#
teacher_preds_soft = teacher_soft_labels

def distillation_loss(y_true_combined, y_pred):

    # <-- Enter your code here: implement hard/soft label separation and weighted loss <--#
    teacher_preds_soft = teacher_soft_labels

y_train_combined = np.concatenate(
    [y_train_cat, teacher_preds_soft],
    axis=1
)

print("Combined label shape:", y_train_combined.shape)



def distillation_loss(y_true_combined, y_pred):
    alpha = 0.5

    y_true_hard = y_true_combined[:, :num_classes]

    y_true_soft = y_true_combined[:, num_classes:]

    hard_loss = tf.keras.losses.categorical_crossentropy(
        y_true_hard,
        y_pred
    )

    soft_loss = tf.keras.losses.categorical_crossentropy(
        y_true_soft,
        y_pred
    )

    total_loss = alpha * hard_loss + (1 - alpha) * soft_loss

    return total_loss

Combined label shape: (124, 6)


In [31]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

# <-- Enter your code here <--#
student_model.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy']
)

history_student = student_model.fit(
    X_train_scaled,
    y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2
)

Epoch 1/10
13/13 [==============================] - 1s 23ms/step - loss: 1.0739 - accuracy: 0.3434 - val_loss: 1.0506 - val_accuracy: 0.5200
Epoch 2/10
13/13 [==============================] - 0s 6ms/step - loss: 0.8806 - accuracy: 0.7172 - val_loss: 0.8906 - val_accuracy: 0.6800
Epoch 3/10
13/13 [==============================] - 0s 7ms/step - loss: 0.7335 - accuracy: 0.8081 - val_loss: 0.7579 - val_accuracy: 0.7600
Epoch 4/10
13/13 [==============================] - 0s 7ms/step - loss: 0.6113 - accuracy: 0.9192 - val_loss: 0.6445 - val_accuracy: 0.8000
Epoch 5/10
13/13 [==============================] - 0s 7ms/step - loss: 0.5082 - accuracy: 0.9495 - val_loss: 0.5363 - val_accuracy: 0.8400
Epoch 6/10
13/13 [==============================] - 0s 6ms/step - loss: 0.4139 - accuracy: 0.9596 - val_loss: 0.4440 - val_accuracy: 0.8800
Epoch 7/10
13/13 [==============================] - 0s 7ms/step - loss: 0.3363 - accuracy: 0.9697 - val_loss: 0.3650 - val_accuracy: 0.9600
Epoch 8/10
13/13 [=

In [32]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

# <-- Enter your code here <--#
converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
kd_tflite_model = converter.convert()

with open("model_kd.tflite", "wb") as f:
    f.write(kd_tflite_model)

kd_model_size_kb = os.path.getsize("model_kd.tflite") / 1024

print(f"Knowledge Distillation TFLite model size: {kd_model_size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmptd301veg/assets


INFO:tensorflow:Assets written to: /tmp/tmptd301veg/assets
2026-05-12 23:01:46.516864: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-12 23:01:46.516950: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.


Knowledge Distillation TFLite model size: 6.10 KB


2026-05-12 23:01:46.517279: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmptd301veg
2026-05-12 23:01:46.518410: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-12 23:01:46.518435: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmptd301veg
2026-05-12 23:01:46.522474: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-12 23:01:46.582728: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmptd301veg
2026-05-12 23:01:46.602508: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 85239 microseconds.


In [33]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#

y_pred_probs = student_model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_probs, axis=1)

y_true = np.argmax(y_test_cat, axis=1)

accuracy = np.mean(y_pred == y_true)
print("Student Model Accuracy:", accuracy)


print("\nClassification Report:")
print(classification_report(y_true, y_pred))


print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

2/2 [==============================] - 0s 6ms/step
Student Model Accuracy: 0.9629629629629629

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.95      0.95        19
           1       0.95      1.00      0.98        21
           2       1.00      0.93      0.96        14

    accuracy                           0.96        54
   macro avg       0.97      0.96      0.96        54
weighted avg       0.96      0.96      0.96        54


Confusion Matrix:
[[18  1  0]
 [ 0 21  0]
 [ 1  0 13]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Yes, the model size can be reduced further by combining multiple compression techniques together, such as applying INT8 quantization to the smaller knowledge-distilled student model. This reduces the number of parameters and stores them using fewer bits, creating a smaller TFLite model. However, additional compression may slightly reduce accuracy, so there is a tradeoff between model size and performance.


Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

Among all models, the INT8 quantized model had the smallest size at 5.74 KB and also achieved the best performance with 100% accuracy on the test set. The baseline, dynamic range quantized, float16 quantized, and pruned models all achieved about 98.15% accuracy, while the knowledge distillation student model achieved slightly lower accuracy at 96.30%. Overall, INT8 quantization provided the best balance between small model size and high classification performance.


2. **Propose a strategy** that combines or enhances techniques learned so far.

A good strategy is to combine knowledge distillation with INT8 quantization. First, train a smaller student model using the baseline model as the teacher, then apply full integer quantization to the student model. This should reduce size more than using the baseline model alone, while still keeping reasonable accuracy.


4. **Implement** your proposed solution.

5. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

6. **Justify your results:**
   - If further size reduction is not possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size further, highlight what change made the biggest difference.

I was able to reduce the model size further by combining knowledge distillation with INT8 quantization. The biggest improvement came from using a smaller student model before applying quantization, which reduced the number of parameters while still keeping good accuracy. The final KD + INT8 model reached a size of 3.62 KB with 96.30% accuracy, showing that additional compression is possible with only a small decrease in performance.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [35]:
# Combine Knowledge Distillation + INT8 Quantization

converter = tf.lite.TFLiteConverter.from_keras_model(student_model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

kd_int8_tflite_model = converter.convert()

with open("model_kd_int8.tflite", "wb") as f:
    f.write(kd_int8_tflite_model)

kd_int8_size_kb = os.path.getsize("model_kd_int8.tflite") / 1024

print(f"KD + INT8 TFLite model size: {kd_int8_size_kb:.2f} KB")




# Evaluate KD + INT8 TFLite model

interpreter = tf.lite.Interpreter(model_path="model_kd_int8.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

input_index = input_details[0]["index"]
output_index = output_details[0]["index"]

input_scale, input_zero_point = input_details[0]["quantization"]
output_scale, output_zero_point = output_details[0]["quantization"]

y_pred = []

for sample in X_test_scaled:
    sample = sample.reshape(1, -1).astype(np.float32)

    # Quantize input
    sample_int8 = sample / input_scale + input_zero_point
    sample_int8 = np.round(sample_int8).astype(np.int8)

    interpreter.set_tensor(input_index, sample_int8)
    interpreter.invoke()

    output = interpreter.get_tensor(output_index)

    # Dequantize output
    output = output_scale * (output.astype(np.float32) - output_zero_point)

    pred_class = np.argmax(output, axis=1)[0]
    y_pred.append(pred_class)

y_true = np.argmax(y_test_cat, axis=1)

accuracy = np.mean(np.array(y_pred) == y_true)

print(f"KD + INT8 Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

INFO:tensorflow:Assets written to: /tmp/tmpvkro8mtc/assets


INFO:tensorflow:Assets written to: /tmp/tmpvkro8mtc/assets
/home/mishrmuk/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


KD + INT8 TFLite model size: 3.62 KB
KD + INT8 Accuracy: 0.9630

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.95      0.95        19
           1       0.95      1.00      0.98        21
           2       1.00      0.93      0.96        14

    accuracy                           0.96        54
   macro avg       0.97      0.96      0.96        54
weighted avg       0.96      0.96      0.96        54


Confusion Matrix:
[[18  1  0]
 [ 0 21  0]
 [ 1  0 13]]


2026-05-12 23:11:58.288646: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-12 23:11:58.288738: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-12 23:11:58.289181: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpvkro8mtc
2026-05-12 23:11:58.290581: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-12 23:11:58.290619: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpvkro8mtc
2026-05-12 23:11:58.294729: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-12 23:11:58.352627: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpvkro8mtc
2026-05-12 23:11:58.366309: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 77064 m

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
